# Stage A - Multinomial Logistic Regression from scratch
Baseline model for CIFAR-10 using flattened pixels.

In [ ]:
# Portable project setup. Works after cloning GitHub repo or extracting the ZIP.
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
# If Colab opens the notebook from another location, uncomment and edit this line:
# PROJECT_ROOT = Path("/content/MLFinalProject")
sys.path.insert(0, str(PROJECT_ROOT))
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
for d in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from src.data import get_preprocessed_flat, CLASSES, set_seeds
from src.logreg import SoftmaxRegression
from src.metrics import compute_metrics, classification_report_df, confusion_matrix_array, per_class_accuracy
from src import viz
set_seeds(42)

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test), stats = get_preprocessed_flat()
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("Model parameters:", X_train.shape[1] * 10 + 10)

In [ ]:
# Quick but reproducible grid search. Increase sample_size for final full run if time allows.
sample_size = min(10000, len(X_train))
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), sample_size, replace=False)
X_sub, y_sub = X_train[idx], y_train[idx]
search_space = [
    {"lr":0.1, "lam":0.0, "bs":128},
    {"lr":0.01, "lam":1e-4, "bs":128},
    {"lr":0.001, "lam":1e-4, "bs":64},
]
rows=[]
for hp in search_space:
    m = SoftmaxRegression(X_train.shape[1], 10, lr=hp["lr"], lam=hp["lam"], seed=42)
    hist = m.fit(X_sub, y_sub, X_val, y_val, batch_size=hp["bs"], max_epochs=25, patience=6, verbose=False)
    rows.append({**hp, "val_acc": max(hist["val_acc"])})
    print(hp, "val_acc=", round(max(hist["val_acc"]), 4))
search_df = pd.DataFrame(rows).sort_values("val_acc", ascending=False)
display(search_df)
search_df.to_csv(TABLE_DIR / "stageA_grid_search.csv", index=False)
best = search_df.iloc[0].to_dict()

In [ ]:
model = SoftmaxRegression(X_train.shape[1], 10, lr=float(best["lr"]), lam=float(best["lam"]), seed=42)
history = model.fit(X_train, y_train, X_val, y_val, batch_size=int(best["bs"]), max_epochs=80, patience=10, verbose=True)
y_pred = model.predict(X_test)
metrics = compute_metrics(y_test, y_pred, model_name="Logistic Regression (scratch)")
print(metrics)
report_df = classification_report_df(y_test, y_pred)
display(report_df)
report_df.to_csv(TABLE_DIR / "stageA_classification_report.csv")

In [ ]:
viz.plot_loss_curves(history, title="Stage A Logistic Regression", save_name="stageA_loss_curves.png")
cm = confusion_matrix_array(y_test, y_pred)
viz.plot_confusion_matrix(cm, title="Stage A Logistic Regression Confusion Matrix", save_name="stageA_confusion_matrix.png")

In [ ]:
result_record = {
    "model": "Logistic Regression (scratch)",
    "stage": "A",
    "hyperparameters": {"lr": float(best["lr"]), "lam": float(best["lam"]), "batch_size": int(best["bs"])},
    "n_params": int(X_train.shape[1] * 10 + 10),
    "test_accuracy": float(metrics["accuracy"]),
    "macro_f1": float(metrics["macro_f1"]),
    "weighted_f1": float(metrics["weighted_f1"]),
}
with open(TABLE_DIR / "stageA_result.json", "w") as f: json.dump(result_record, f, indent=2)
print("Stage A result saved.")

## Stage A interpretation
Logistic regression is a useful baseline because it is simple and explains what can be learned from flattened pixel values only. It performs above the 10 percent random baseline, but it is limited because it uses a linear decision boundary and does not know which pixels are neighbours. This is why visually similar classes such as cat, dog, deer, and horse are expected to be confused.